# 04_labeling.ipynb - Fase 4: Triple Barrier Labeling

**Objetivo**: Etiquetar cada entrada potencial con {0: SL, 1: TP, 2: Timeout}

Regla de desempate: Si High toca TP Y Low toca SL en misma vela → SL gana (conservador).

In [8]:
import sys
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA, FEATURES, RISK, BACKTEST
from src.data import load_ohlc_from_yfinance
from src.features import add_all_features
from src.labeling import apply_triple_barrier_to_dataset, create_labeled_dataset

print("Imports listos")

Imports listos


## Labeling para cada ticker admitido

In [9]:
# Ejecutar este bloque con los tickers admitidos
# output: data/labeled/*.csv

features_dir = REPO_ROOT / "data" / "features"
labeled_dir = REPO_ROOT / "data" / "labeled"
labeled_dir.mkdir(parents=True, exist_ok=True)

admitted_df = pd.read_csv(REPO_ROOT / "data" / "universe_admitted.csv")

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    csl = row['csl']
    print(f"\nProcesando {ticker} (Csl={csl:.3f})")

    feature_path = features_dir / f"features_{ticker}.csv"
    if feature_path.exists():
        df = pd.read_csv(feature_path, parse_dates=['Datetime'])
        print(f"  Cargado features desde {feature_path}")
    else:
        print(f"  ⚠️ Features no encontrados en {feature_path}. Calculando en tiempo real.")
        try:
            df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
        except Exception as e:
            print(f"  ❌ Error al cargar {ticker}: {e}")
            continue
        df = add_all_features(df)
        feature_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(feature_path, index=False)
        print(f"  Guardado features en {feature_path}")

    if df.empty:
        print("  ❌ No hay datos disponibles")
        continue

    labeled = create_labeled_dataset(
        df,
        csl,
        tp_sl_ratio=BACKTEST.tp_sl_ratio,
        max_bars=BACKTEST.max_bars
    )

    if labeled.empty:
        print("  ⚠️ No se generaron etiquetas para este ticker")
        continue

    print(f"  ✅ Muestras etiquetadas: {len(labeled)}")
    print(f"  Distribución: {labeled['label'].value_counts().to_dict()}")

    output_path = labeled_dir / f"labeled_{ticker}.csv"
    labeled.to_csv(output_path, index=False)
    print(f"  Guardado: {output_path}")





Procesando MCRB (Csl=2.073)
  Cargado features desde /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/features/features_MCRB.csv
  ✅ Muestras etiquetadas: 3637
  Distribución: {0: 1893, 1: 1171, 2: 573}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled/labeled_MCRB.csv

Procesando LRMR (Csl=2.187)
  Cargado features desde /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/features/features_LRMR.csv
  ✅ Muestras etiquetadas: 5032
  Distribución: {0: 2754, 1: 1759, 2: 519}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/labeled/labeled_LRMR.csv

Procesando ABSI (Csl=2.297)
  Cargado features desde /home/odoo-01/Escritorio/Projecto_Machine_Learning_Quant/smallcap-quant-ml/data/features/features_ABSI.csv
  ✅ Muestras etiquetadas: 5032
  Distribución: {0: 2428, 1: 2057, 2: 547}
  Guardado: /home/odoo-01/Escritorio/Projecto_Machine_Learnin